# Topic Modeling

To discover the most common topics talked about by customers, we will implement topic modeling algorithms.
Specifically, we will use and compare BERTopic (https://maartengr.github.io/BERTopic/index.html) and FASTopic (https://github.com/bobxwu/FASTopic). BERTopic creates contextual review embeddings using a Transformer model, reduces their dimensionality, groups semantically related data into clusters, and uses class-based TF-IDF to identify the words that best represent each topic. FASTopic is a neural topic model that uses pretrained Transformer embeddings and optimal transport to learn relationships among documents, topics, and words. It is designed to provide efficient, stable, and interpretable topic discovery. After evaluation, the selected model will generate the recurring topics. Examples would include food quality, customer service, cleanliness, pricing, atmosphere, and wait times. These results will be displayed in the dashboard as ranked topic bars, topic-frequency charts, or lists of representative keywords.

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [ ]:
import cuml
print(cuml.__version__)

from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN
print("cuML imports working")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

In [ ]:
!pip install bertopic fastopic --quiet

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import time
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from fastopic import FASTopic
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

In [ ]:
df = pd.read_csv("yelp_reviews_clean_CA.csv")
bn_df = pd.read_csv("yelp_academic_dataset_business.csv")

df = df.merge(
    bn_df[["business_id", "name"]],
    on="business_id",
    how="left"
)
print(f"Total rows: {len(df)}")
print(f"Rows with no matching name: {df['name'].isna().sum()}")
df.head()

In [ ]:
# DONT RUN THIS ONE, HOLDS THE ~7 MIL ROWS CSV
chunks = []
for chunk in pd.read_csv("yelp_reviews_clean.csv", chunksize=5000):
     chunks.append(chunk)
df = pd.concat(chunks, ignore_index=True)

bn_df = pd.read_csv("yelp_academic_dataset_business.csv")

df = df.merge(
    bn_df[["business_id", "name"]],
    on="business_id",
    how="left"
)
print(f"Total rows: {len(df)}")
print(f"Rows with no matching name: {df['name'].isna().sum()}")
df.head()

In [ ]:
business_sizes = df.groupby("business_id").size()
print(f"Total unique businesses: {len(business_sizes)}")
print(business_sizes.describe())
print(f"\nBusinesses with 1 review: {(business_sizes == 1).sum()}")
print(f"Businesses with 10+ reviews: {(business_sizes >= 10).sum()}")

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# BERTopic

In [ ]:
bertopic_results_dir = "bertopic_results"
os.makedirs(bertopic_results_dir, exist_ok=True)

def compute_coherence(topic_words_list, tokenized_docs):
    if len(topic_words_list) == 0:
        return None
    dictionary = Dictionary(tokenized_docs)
    cm = CoherenceModel(
        topics=topic_words_list,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

business_groups = df.groupby("business_id")
total_businesses = business_groups.ngroups
print(f"Total businesses to process (BERTopic): {total_businesses}\n")

bertopic_failed = []

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=10)

In [ ]:
start_all = time.time()

import itertools

start_index = 2600  # 0-indexed, so this skips the first 2600 businesses

business_groups_list = list(business_groups)  # materializes (biz_id, group) pairs
total_businesses = len(business_groups_list)

MIN_REVIEWS_FOR_TOPIC_MODELING = 15
for biz_num, (biz_id, group) in enumerate(itertools.islice(business_groups_list, start_index, None),
    start=start_index + 1):
    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]
    result_path = f"{bertopic_results_dir}/{biz_id}.pkl"

    if os.path.exists(result_path):
        print(f"[{biz_num}/{total_businesses}] {biz_id}: already done, skipping")
        continue

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): only {len(biz_docs)} reviews, skipping (below minimum)")
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id,
                "business_name": biz_name,
                "num_reviews": len(biz_docs),
                "skipped": True,
                "reason": "insufficient_reviews"
            }, f)
        continue

    print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): {len(biz_docs)} reviews")

    try:
        biz_embeddings = embedding_model.encode(biz_docs, batch_size=96, show_progress_bar=False)

        bertopic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model,
    hdbscan_model=hdbscan_model, calculate_probabilities=False, verbose=False)
        bertopic_topics, _ = bertopic_model.fit_transform(biz_docs, biz_embeddings)

        topic_info = bertopic_model.get_topics()
        bertopic_topic_words = [[w for w, _ in words] for tid, words in topic_info.items() if tid != -1]

        tokenized = [d.lower().split() for d in biz_docs]
        bertopic_coherence = compute_coherence(bertopic_topic_words, tokenized)

        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id,
                "business_name": biz_name,
                "num_reviews": len(biz_docs),
                "topics": bertopic_topics,
                "topic_words": bertopic_topic_words,
                "coherence": bertopic_coherence,
            }, f)

    except Exception as e:
        print(f"  FAILED: {e}")
        bertopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))
        continue

elapsed = time.time() - start_all
print(f"\nBERTopic done. {len(bertopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

In [ ]:
bertopic_failures_df = pd.DataFrame(bertopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
bertopic_failures_df.sort_values("num_reviews").head(20)

# FASTopic

In [ ]:
fastopic_results_dir = "fastopic_results"
os.makedirs(fastopic_results_dir, exist_ok=True)
fastopic_failed = []

In [ ]:
import itertools
import logging
logging.getLogger("fastopic").setLevel(logging.WARNING)

start_index = 2600  # 0-indexed, so this skips the first 2600 businesses

business_groups_list = list(business_groups)  # materializes (biz_id, group) pairs
total_businesses = len(business_groups_list)

start_all = time.time()

MIN_REVIEWS_FOR_TOPIC_MODELING = 15
for biz_num, (biz_id, group) in enumerate(itertools.islice(business_groups_list, start_index, None), start=start_index + 1):
    result_path = f"{fastopic_results_dir}/{biz_id}.pkl"

    if os.path.exists(result_path):
        print(f"[{biz_num}/{total_businesses}] {biz_id}: already done, skipping")
        continue

    biz_docs = group["text"].astype(str).tolist()
    biz_name = group["name"].iloc[0]

    if len(biz_docs) < MIN_REVIEWS_FOR_TOPIC_MODELING:
        print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): only {len(biz_docs)} reviews, skipping (below minimum)")
        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id,
                "business_name": biz_name,
                "num_reviews": len(biz_docs),
                "skipped": True,
                "reason": "insufficient_reviews"
            }, f)
        continue

    print(f"[{biz_num}/{total_businesses}] {biz_id} ({biz_name}): {len(biz_docs)} reviews")

    try:
        # Match topic count to what BERTopic found for this business, if available, else default
        bertopic_path = f"{bertopic_results_dir}/{biz_id}.pkl"
        if os.path.exists(bertopic_path):
            with open(bertopic_path, "rb") as f:
                bertopic_result = pickle.load(f)
            num_topics_guess = max(len(bertopic_result["topic_words"]), 2)
        else:
            num_topics_guess = 5

        fastopic_model = FASTopic(num_topics=num_topics_guess, verbose=False)
        fastopic_topic_words, fastopic_doc_topics = fastopic_model.fit_transform(biz_docs)

        tokenized = [d.lower().split() for d in biz_docs]
        fastopic_coherence = compute_coherence(fastopic_topic_words, tokenized)

        with open(result_path, "wb") as f:
            pickle.dump({
                "business_id": biz_id,
                "business_name": biz_name,
                "num_reviews": len(biz_docs),
                "topic_words": fastopic_topic_words,
                "doc_topics": fastopic_doc_topics,
                "coherence": fastopic_coherence,
            }, f)

    except Exception as e:
        print(f"  FAILED: {e}")
        fastopic_failed.append((biz_id, biz_name, len(biz_docs), str(e)))
        continue

elapsed = time.time() - start_all
print(f"\nFASTopic done. {len(fastopic_failed)} failed out of {total_businesses}. Took {elapsed/60:.1f} min")

In [ ]:
fastopic_failures_df = pd.DataFrame(fastopic_failed, columns=["business_id", "business_name", "num_reviews", "error"])
fastopic_failures_df.sort_values("num_reviews").head(20)

In [ ]:
comparison_rows = []

for fname in os.listdir(bertopic_results_dir):
    biz_id = fname.replace(".pkl", "")
    bertopic_path = f"{bertopic_results_dir}/{fname}"
    fastopic_path = f"{fastopic_results_dir}/{fname}"

    if not os.path.exists(fastopic_path):
        continue

    with open(bertopic_path, "rb") as f:
        b = pickle.load(f)
    with open(fastopic_path, "rb") as f:
        fst = pickle.load(f)

    comparison_rows.append({
        "business_id": biz_id,
        "business_name": b["business_name"],
        "num_reviews": b["num_reviews"],
        "bertopic_coherence": b["coherence"],
        "fastopic_coherence": fst["coherence"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df